In [44]:
import pandas as pd

df = pd.read_csv('dataset.csv')
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [4]:
gtin_status = df["gtin"].isna().map({True: "GTIN missing", False: "GTIN present"})

summary = gtin_status.value_counts().rename_axis("gtin_status").reset_index(name="row_count")
summary["pct"] = (summary["row_count"] / len(df) * 100).round(1)

summary

,gtin_status,row_count,pct
0,GTIN missing,41545,58.0
1,GTIN present,30078,42.0


Highly Imbalanced dataset already?

In [ ]:
gtin_df = df.dropna(subset=["gtin"]).copy()
gtin_df["gtin"] = gtin_df["gtin"].astype("Int64").astype(str)
grouped = gtin_df.groupby("gtin").agg(
    n_rows=("sku_id", "count"),
    n_distinct_names=("sku_name_eng", "nunique"),
    n_distinct_brands=("brand", "nunique"),
    n_distinct_categories=("category", "nunique"),
).reset_index()
dirty = grouped[grouped["n_distinct_names"] > 1].sort_values("n_distinct_names", ascending=False)

print(f"Total unique GTINs: {grouped.shape[0]}")
print(f"GTINs mapped to more than one distinct product name: {dirty.shape[0]}")
print(f"Share of GTINs that are reused: {dirty.shape[0] / grouped.shape[0]:.1%}")



Total unique GTINs: 14997
GTINs mapped to more than one distinct product name: 6461
Share of GTINs that are reused: 43.1%


In [10]:
example = gtin_df[gtin_df["gtin"] == "8414100000013"][
    ["sku_id", "gtin", "sku_name_eng", "brand", "category", "retailer", "country"]
].sort_values("brand")

example

,sku_id,gtin,sku_name_eng,brand,category,retailer,country
20373,160599962,8414100000013,Gaseosoa the landlady bottle 50 centiliters,La Casera,Carbonated Bottled Water,Alcampo,Spain
33226,381213417,8414100000013,soda of orange the landlady can 33 centiliters,La Casera,Orange Carbonates,Alcampo,Spain
20431,160792691,8414100000013,refreshing drink of black tea and peach MAY TE...,May Tea,Still RTD Tea,Alcampo,Spain
20442,160822875,8414100000013,drink green tea and refreshing lemon MAY TEA 1 l.,May Tea,Still RTD Tea,Alcampo,Spain
20371,160595886,8414100000013,soft drink cooled Florida SUNNY DELIGHT FLORID...,Sunny Delight,Functional Bottled Water,Alcampo,Spain
20398,160673819,8414100000013,soft drink cooled of strawberry SUNNY DELIGHT ...,Sunny Delight,Juice Drinks (up to 24% Juice),Alcampo,Spain
20448,160855794,8414100000013,soft drink cooled with taste of berries SUNNY ...,Sunny Delight,Other Non-Cola Carbonates,Alcampo,Spain


There we have it, same GTIN used for unrelated products. Ay caramba.  
We do expect to see some naming variations but totally unrelated products with the same GTIN is a big problem for us. Our imbalanced data just got worse.

In [13]:
cross_brand = grouped[grouped["n_distinct_brands"] > 1]

print(f"GTINs shared across more than one brand: {cross_brand.shape[0]}")
print(f"Share of all populated GTINs: {cross_brand.shape[0] / grouped.shape[0]:.1%}")




cross_brand_gtins = cross_brand["gtin"].tolist()

retailer_breakdown = gtin_df[gtin_df["gtin"].isin(cross_brand_gtins)].groupby("retailer")["gtin"].nunique().sort_values(ascending=False)

retailer_breakdown[:5]

GTINs shared across more than one brand: 87
Share of all populated GTINs: 0.6%


retailer
Hy-Vee         41
safeway        33
amazon         32
Giant Eagle    30
H-E-B          26
Name: gtin, dtype: int64

Ok, it's systemic and across several retailers. Ciesta saved.

In [8]:
gold_candidates = grouped.merge(
    gtin_df.groupby("gtin")["retailer"].nunique().rename("n_distinct_retailers"),
    on="gtin"
)

gold = gold_candidates[
    (gold_candidates["n_distinct_names"] == 1) &
    (gold_candidates["n_distinct_brands"] == 1) &
    (gold_candidates["n_distinct_categories"] == 1) &
    (gold_candidates["n_distinct_retailers"] > 1)
]

print(f"'Gold' GTINs: {gold.shape[0]} / {gold_candidates.shape[0]} ({gold.shape[0]/gold_candidates.shape[0]:.1%})")

'Gold' GTINs: 1046 / 14997 (7.0%)


In [9]:
top_gold_gtins = gold.sort_values("n_distinct_retailers", ascending=False).head(10)["gtin"]

gold_examples = gtin_df[gtin_df["gtin"].isin(top_gold_gtins)][
    ["gtin", "sku_id", "sku_name_eng", "brand", "category", "retailer", "country"]
].sort_values(["gtin", "retailer"])

gold_examples

,gtin,sku_id,sku_name_eng,brand,category,retailer,country
42247,6410270005584,524366670,marjex cold pressed cranberry juice 0.5l,Marjex,Not from Concentrate 100% Juice,Food Market Herkku,Finland
17380,6410270005584,125345346,marjex cold pressed cranberry juice 0.5l,Marjex,Not from Concentrate 100% Juice,K Ruoka,Finland
18291,6410270005584,143513794,marjex cold pressed cranberry juice 0.5l,Marjex,Not from Concentrate 100% Juice,S-kaupat,Finland
27470,6410270005584,265021537,marjex cold pressed cranberry juice 0.5l,Marjex,Not from Concentrate 100% Juice,kauppahalli24.fi,Finland
42064,6410270005591,524159138,marjex cold pressed sea buckthorn juice 0.5l,Marjex,Not from Concentrate 100% Juice,Food Market Herkku,Finland
16959,6410270005591,119287562,marjex cold pressed sea buckthorn juice 0.5l,Marjex,Not from Concentrate 100% Juice,K Ruoka,Finland
35547,6410270005591,421343443,marjex cold pressed sea buckthorn juice 0.5l,Marjex,Not from Concentrate 100% Juice,S-kaupat,Finland
27455,6410270005591,264865814,marjex cold pressed sea buckthorn juice 0.5l,Marjex,Not from Concentrate 100% Juice,kauppahalli24.fi,Finland
41951,6415130086338,524034502,marli vital crush + 10 vitamins 2 dl,Marli,Nectars,Food Market Herkku,Finland
33888,6415130086338,394385354,marli vital crush + 10 vitamins 2 dl,Marli,Nectars,K Ruoka,Finland


The rest are truly gold, or are they?

In [18]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(gtin_df[gtin_df["gtin"] == "5018076100697"][["sku_id", "sku_name_eng"]])

,sku_id,sku_name_eng
12120,78520123,sunita | Lemon Juice - organic | 2 x 12 x 250ml
12149,78624931,sunita | Lemon Juice - organic | 2 x 250ml
12150,78627530,sunita | Lemon Juice - organic | 7 x 250ml
12345,79312596,sunita | Lemon Juice - organic | 4 x 250ml
12373,79454137,sunita | Lemon Juice - organic | 11 x 250ml
12474,79815415,sunita | Lemon Juice - organic | 1 x 250ml
12955,81410103,sunita | Lemon Juice - organic | 1 x 250ml
12969,81481708,sunita | Lemon Juice - organic | 3 x 250ml
12991,81539593,sunita | Lemon Juice - organic | 8 x 250ml
13126,81999104,sunita | Lemon Juice - organic | 10 x 250ml


# EuromonitoR


# My Approach
 This dataset required more of a judgment call rather than anything else:
 - How to define ground truth?  
 - I need more ground truth

Which GTINs do I use? How do I define a GTIN?

The rest is fairly straightforward; regex and semantic transformer model:

I didnt want to use REGEX to extract as much as possible, since it's brittle and not an elegant solution, but as ive done before, I ended up back with a lookup dictionary as a starting point. For the first submission, I chose to touch as many bases as possible instead on focusing on just REGEX extraction. 
  
  
Since REGEX is deterministic, we get a clear picture of what the semantic sentence transformer can do and serves as a solid starting point. The additional experiments would be further finetuning the gates, or regex to extract enough but not too much.

Since the imbalance was around 1:17, I masked 1:1 to augment data, resulting in an almost 1:1. While we're talking about our data situtation, Ive already REGEX'ed as much as I could, the now hard negatives and positives serve for OnlineContrastiveLoss, Tripletloss, and MultipleNegativesRakingLoss. 

Of course theres a lot more room for some gains, but hopefully with all ive done, you guys have a clear picture of what i do. 


# In a Perfect World:
- TruncatedSVD
- Multi-vector representations -ColBERT-
- Better pooling strategies
- NMF
- WandB
- Topic modeling
- Fuzzy string matching
- Embedding-based similarity search
- Text classification
- More experiments loss function / model payload
- Custom semantic transformer model
- Minimize larger semantic transformer model
- Pure Embedding Clustering
- Graph‑Based Entity Resolution
- LLM as a judge 
- Hybrid Models
- Error Analysis
- Ablaition analysis
- Testing
- Operational Performance (Latency, Cost).
- Model API endpoint, Model card, Prom/Graf/ Pandera/ GE/ Pydantic.
- Stress test model, alert + retraining.
- Model calibration.
- Embedding versioning + reindex cost.
- Optimize model/data for cost/performance.
- Canary deployment



# Coding Conventions:
- SSOT (configs.*)
- Factory Pattern
- Deterministic Checks (idempotency/pure functions)
- Reproducibility (Docker)
- Data transparency / traceability (model payload)

# Random Findings
GTIN	58% missing; of the 42% populated, 43.1% are non-unique; 0.6% shared across different brands	  

Volume Generally reliable (92%+) — but a placeholder value (200) overwrites sizes from 2 to 96 fl oz flattened to "200." for at least 3,247 rows	Concentrated in a subset (coffee/cold brew).



  






[Entity Matching]

Business Case:  
Wrong metrics will invalidate all downstream economic studies. Economical inaccuracies on my watch? Think again.


## Core problem

Same physical product with GTIN (barcode/ Ground Truth) but many are missing/invalid/reused, one GTIN
can carry inconsistent attributes across retailers, and different GTINs can describe the same product. 

## Approach

1. **Validate GTINs** (length, check digit) — clean vs noisy barcodes.
2. **Extract attributes** (volume, pack, flavor, type) from titles + fields.
3. **Deterministic three-way gate** on volume/pack/flavor: block impossible
   matches (hard_no), route uncertain ones to fallback, send likely
   duplicates to embeddings (proceed).
4. **Fine-tune an embedding model** on cleaned text (NO numbers — sizes are
   the gate's job, never the model's) to learn product identity.
5. **Component-fold evaluation** — no barcode straddles a split boundary;
   metrics are honest (PR-AUC primary, F1 at a fixed threshold).

  